# Topic modeling

## 0. Libraries import

In [1]:
import pandas as pd
import numpy as np
import ast
import gensim
import gensim.corpora as corpora
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import nltk
import string 
import re 

In [2]:
from gensim.models import LdaModel
from gensim.corpora import Dictionary
from gensim.models import LdaMulticore
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
from nltk.stem import WordNetLemmatizer
from collections import Counter
from gensim.models import CoherenceModel
from sklearn.linear_model import LogisticRegression

In [3]:
nltk.download('punkt_tab')
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\simon\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\simon\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\simon\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

## 1. Data import

In [8]:
df = pd.read_csv('Kickstarter_processed.csv', index_col=0)

df["description_filtered"] = df["description_filtered"].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)

## 2. Topic modeling

#### 2. Bag of Words creation

In [9]:
df_film = df[
    (df["category"] == "Film & Video") &
    (df["description_filtered"].notna())
].copy()

texts_film = df_film["description_filtered"].tolist()

dictionary_film = Dictionary(texts_film)
corpus_film = [dictionary_film.doc2bow(text) for text in texts_film]

print("\nNumber of words in the dictionary:", len(dictionary_film))
print("Number of documents:", len(corpus_film))


Number of words in the dictionary: 28268
Number of documents: 2011


#### 3. Latent Dirichlet Allocation (LDA) and topic selection

1) In order to determine the optimal number of topics, several LDA models were trained with the number of topics ranging from 2 to 7.

2) For each model, the coherence score was computed to evaluate the semantic consistency of the extracted topics and the results show that the highest score is achieved when the number of topics is 3. Therefore, the model with 3 topics produces the most coeherent and interpretable topics.

3) However, the overall topic quality is still influenced by the presence of generic and noisy terms in the corpus, suggesting that further text preprocessing could improve the results.

In [10]:
lda_film = LdaMulticore(
    corpus = corpus_film,
    id2word = dictionary_film,
    num_topics = 3,
    passes = 10,
    workers = 2,
    random_state = 11
)

coherence_model = CoherenceModel(
    model = lda_film,
    texts = texts_film,
    dictionary = dictionary_film,
    coherence = 'c_v'
)

print(coherence_model.get_coherence())

print("\nTopics:")
for i, topic in lda_film.print_topics(num_words=10):
    print(f"Topic {i}: {topic}")

0.32773170195286433

Topics:
Topic 0: 0.005*"festival" + 0.003*"producer" + 0.003*"movie" + 0.003*"short" + 0.003*"documentary" + 0.003*"community" + 0.003*"family" + 0.003*"feature" + 0.002*"filmmaker" + 0.002*"experience"
Topic 1: 0.006*"reward" + 0.004*"show" + 0.004*"animation" + 0.004*"character" + 0.004*"episode" + 0.003*"series" + 0.003*"movie" + 0.003*"campaign" + 0.003*"video" + 0.003*"sound"
Topic 2: 0.004*"producer" + 0.004*"short" + 0.003*"show" + 0.003*"art" + 0.003*"festival" + 0.003*"music" + 0.003*"feature" + 0.003*"actor" + 0.003*"set" + 0.003*"crew"


#### 4. Relationship between topics and project success

1. In order to analyze the relationship between topics and project success, each document was assigned to its dominant topic based on the highest probability returned by the LDA model. 

2. For each topic, the average value of the status variable (0 = failure, 1 = success) was computed, allowing the estimation of the success rate associated with each topic and then the number of projects per topic was calculated.

4. The results show that some topics are associated with higher success rates than others, suggesting that the thematic content of a project may influence its likelihood of success. However, further statistical testing would be required to determine whether these differences are statistically significant.

In [11]:
doc_topics = []

for doc in corpus_film:
    topics = lda_film.get_document_topics(doc)
    dominant_topic = max(topics, key=lambda x: x[1])[0]
    doc_topics.append(dominant_topic)

df_film = df[df['category'] == 'Film & Video'].copy()
df_film['topic'] = doc_topics

summary = df_film.groupby('topic').agg({
    'status': ['mean', 'count']
})

summary.columns = ['success_rate', 'num_projects']

print(summary)

       success_rate  num_projects
topic                            
0          0.433014           836
1          0.439065           599
2          0.460069           576


#### 5. Logistic regression

1. To further investigate the relationship between topics and project success, a logistic regression model was applied using the dominant topic (Topic 0) as a categorical predictor. However, since the topic variable is categorical, one-hot encoding was used to transform it into binary features, with Topic 0 serving as the baseline category.

2. The model was trained using the project status (0 = failure, 1 = success) as the target variable, allowing the estimation of the effect of each topic on the probability of success.

3. The results show that Topic 2 has a strongly negative coefficient, indicating a significantly lower probability of success compared to the baseline topic (Topic 0), while Topic 1 has only a slightly negative coefficient, suggesting a success probability similar to the baseline. Therefore, the regression results confirm the findings from the descriptive analysis, reinforcing the conclusion that the thematic content of a project is associated with its likelihood of success.

In [ ]:
X = pd.get_dummies(df_film['topic'], prefix='topic', drop_first=True)
y = df_film['status']

model = LogisticRegression(max_iter=1000)
model.fit(X, y)

coefficients = pd.DataFrame({
    'feature': X.columns,
    'coefficient': model.coef_[0]
})

print(coefficients)

   feature  coefficient
0  topic_1    -0.072475
1  topic_2    -0.568832


### Games

#### 1. Identification of words appearing in more than % of documents

#### 2. Bag of Word creation

#### 3. LDA

### Music

#### 1. Identification of words appearing in more than % of documents

#### 2. Bag of Word creation

#### 3. LDA

### Publishing

#### 1. Identification of words appearing in more than % of documents

#### 2. Bag of Word creation

#### 3. LDA

### Technology

#### 1. Identification of words appearing in more than % of documents

#### 2. Bag of Word creation

#### 3. LDA